In [ ]:
# Fix: Kaggle's default NumPy 1.26.4 is CORRECT - prevent upgrades to 2.x
import sys
print("Step 1: Record Kaggle's default NumPy version...")
import numpy as np
default_numpy = np.__version__
print(f"   Default NumPy: {default_numpy}")

print("\nStep 2: Install ultralytics...")
import subprocess
subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)

print("\nStep 3: LOCK NumPy back to Kaggle's default...")
subprocess.run(["pip", "install", "-q", "--force-reinstall", f"numpy=={default_numpy}"], check=True)

print("\nInstallation complete!")
import numpy as np
from ultralytics import YOLO
print(f"   NumPy: {np.__version__}")
print("   OK Ultralytics imported successfully")

In [ ]:
# Smoke Test: No GPU, TINY dataset, validates pipeline only
import json
import os
import shutil
import sys
from pathlib import Path

import numpy as np

print("=" * 80)
print("ABLATION 4 SMOKE TEST - CPU ONLY")
print("=" * 80)
print("\nPurpose: Validate all 3 variants with TINY dataset")
print("Expected runtime: 3-5 minutes on CPU")
print("=" * 80)

In [ ]:
# Import modules
if os.path.exists("/kaggle/input/wound-seg-code-v1"):
    sys.path.insert(0, "/kaggle/input/wound-seg-code-v1")
    from data.yolo_config import create_yolo_config
    from evaluation.metrics import (
        calculate_bootstrap_ci,
        calculate_boundary_f1,
        calculate_segmentation_metrics,
    )
    from evaluation.yolo_utils import yolo_result_to_metrics_input
else:
    sys.path.insert(0, str(Path.cwd().parent))
    from src.evaluation.metrics import (
        calculate_segmentation_metrics,
        calculate_boundary_f1,
        calculate_bootstrap_ci,
    )
    from src.data.yolo_config import create_yolo_config
    from src.evaluation.yolo_utils import yolo_result_to_metrics_input

import cv2
from ultralytics import YOLO

print("OK All modules imported")

In [ ]:
# Use baseline-outputs for smoke test (has YOLO structure)
IN_KAGGLE = Path("/kaggle/input").exists()

if IN_KAGGLE:
    YOLO_DATASET = Path("/kaggle/input/baseline-outputs/data_wound_seg")
    OUTPUT_PATH = Path("/kaggle/working")
else:
    YOLO_DATASET = Path.cwd().parent / "simulated_kaggle_data/input/baseline-outputs"
    OUTPUT_PATH = Path.cwd().parent / "kaggle_outputs/ablation_4_smoke"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {YOLO_DATASET}")
print(f"Output: {OUTPUT_PATH}")

# Load first 3 test images for smoke test
test_imgs = sorted((YOLO_DATASET / "test/images").glob("*.png"))[:3]
test_pairs = [(str(img), str(YOLO_DATASET / "test_masks" / img.name)) for img in test_imgs]

print(f"\nTest pairs: {len(test_pairs)} (smoke test subset)")

## Smoke Test 1: DDI Dataset Preparation (Simulated)

**Test**: Validate fixed split logic with mock DDI data

In [ ]:
# Simulate DDI dataset preparation
from sklearn.model_selection import train_test_split

print("\n" + "=" * 80)
print("SMOKE TEST 1: DDI Dataset Preparation")
print("=" * 80)

# Create mock DDI paths (simulate 643 images)
mock_ddi_paths = [Path(f"ddi_image_{i:04d}.png") for i in range(643)]

# Test fixed split (seed=42)
ddi_train_paths, ddi_test_paths = train_test_split(
    mock_ddi_paths, test_size=0.2, random_state=42, shuffle=True
)

print(f"\nMock DDI split (seed=42):")
print(f"  Total: {len(mock_ddi_paths)}")
print(f"  Train: {len(ddi_train_paths)} (expected: 514)")
print(f"  Test: {len(ddi_test_paths)} (expected: 129)")

# Validate
assert len(ddi_train_paths) == 514, f"Train split wrong: {len(ddi_train_paths)}"
assert len(ddi_test_paths) == 129, f"Test split wrong: {len(ddi_test_paths)}"

print("\nOK DDI split validation PASSED")

## Smoke Test 2: Combined Dataset Creation (4a)

In [ ]:
print("\n" + "=" * 80)
print("SMOKE TEST 2: Ablation 4a Dataset Creation")
print("=" * 80)

# Create 4a combined dataset with TINY data
COMBINED_4A = OUTPUT_PATH / "ablation_4a_dataset"
(COMBINED_4A / "train/images").mkdir(parents=True, exist_ok=True)
(COMBINED_4A / "train/labels").mkdir(parents=True, exist_ok=True)
(COMBINED_4A / "test/images").mkdir(parents=True, exist_ok=True)
(COMBINED_4A / "test/labels").mkdir(parents=True, exist_ok=True)

# Copy TINY wound images
tiny_train_imgs = sorted((YOLO_DATASET / "train/images").glob("*.png"))[:4]  # First 4 for smoke test
# Masks not needed for 4a (uses images only)

wound_count = 0
for img_path in tiny_train_imgs:
    shutil.copy(img_path, COMBINED_4A / "train/images" / img_path.name)
    # Create empty label (smoke test - no actual conversion)
    label_path = COMBINED_4A / "train/labels" / f"{img_path.stem}.txt"
    label_path.write_text("0 0.5 0.5 0.6 0.6 0.4 0.4")  # Mock label
    wound_count += 1

# Simulate adding DDI (use 2 extra mock files)
for i in range(2):
    mock_ddi = COMBINED_4A / "train/images" / f"ddi_mock_{i}.png"
    mock_ddi.touch()
    label_path = COMBINED_4A / "train/labels" / f"ddi_mock_{i}.txt"
    label_path.write_text("")  # Empty = hard negative

print(f"\n4a dataset created:")
print(f"  Wounds: {wound_count}")
print(f"  DDI (mock): 2")
print(f"  Total train: {wound_count + 2}")

# Create YOLO config
config_result = create_yolo_config(
    dataset_root=COMBINED_4A,
    train_dir="train",
    val_dir="test",
    test_dir="test",
    output_path=OUTPUT_PATH / "wound_dataset_4a.yaml",
)

assert config_result["success"], f"Config failed: {config_result['error']}"
print(f"\nOK 4a config created: {config_result['data']['config_path']}")

## Smoke Test 3: Schema Validation (All Variants)

In [ ]:
print("\n" + "=" * 80)
print("SMOKE TEST 3: Schema Validation")
print("=" * 80)

# Test schema for metrics calculation
print("\nTesting metrics schema...")

# Create mock masks
pred_mask = np.random.randint(0, 2, (512, 512), dtype=np.uint8)
gt_mask = np.random.randint(0, 2, (512, 512), dtype=np.uint8)

# Calculate metrics
metrics = calculate_segmentation_metrics(pred_mask, gt_mask, "smoke_test_image")

# Validate schema
expected_keys = {"image_id", "dice", "iou", "precision", "recall", "tp", "fp", "tn", "fn"}
actual_keys = set(metrics.keys())

print(f"Expected keys: {expected_keys}")
print(f"Actual keys: {actual_keys}")

assert expected_keys.issubset(
    actual_keys
), f"Schema mismatch! Missing: {expected_keys - actual_keys}"

print("\nOK Metrics schema VALIDATED")

# Test boundary F1
boundary_f1 = calculate_boundary_f1(pred_mask, gt_mask, tolerance=2)
assert isinstance(boundary_f1, float), f"Boundary F1 wrong type: {type(boundary_f1)}"
print(f"OK Boundary F1 calculation works: {boundary_f1:.4f}")

## Smoke Test 4: Completion Marker Logic

In [ ]:
print("\n" + "=" * 80)
print("SMOKE TEST 4: Completion Marker Logic")
print("=" * 80)

# Test completion marker pattern
completion_4a = OUTPUT_PATH / "ablation_4a_completion.json"

# Simulate 4a completion
mock_results = {
    "variant": "4a_mixed",
    "dice_mean": 0.8234,
    "dice_ci_lower": 0.8156,
    "dice_ci_upper": 0.8312,
    "per_image_results": {"seed_42": [{"image_id": "test", "dice": 0.82}]},
}

with open(completion_4a, "w") as f:
    json.dump(mock_results, f, indent=2)

print(f"\nOK Created completion marker: {completion_4a}")

# Test load-or-train logic
if completion_4a.exists():
    with open(completion_4a) as f:
        loaded = json.load(f)
    print(f"OK Loaded existing results: Dice = {loaded['dice_mean']:.4f}")
    print("  (In production, would skip training)")
else:
    print("  (In production, would train from scratch)")

# Validate completion marker schema
required_fields = {"variant", "dice_mean", "dice_ci_lower", "dice_ci_upper", "per_image_results"}
assert required_fields.issubset(set(loaded.keys())), "Completion marker schema wrong"

print("\nOK Completion marker logic VALIDATED")

## Smoke Test 5: YOLO Model Initialization (CPU)

In [ ]:
print("\n" + "=" * 80)
print("SMOKE TEST 5: YOLO Model Initialization")
print("=" * 80)

# Test YOLO model loading (CPU only - no training)
print("\nLoading YOLOv8n-seg (COCO weights)...")
model = YOLO("yolov8n-seg.pt")

print(f"OK Model loaded successfully")
print(f"  Classes: {len(model.names)}")
print(f"  Device: {model.device}")

# Test inference on TINY image (CPU)
if len(test_pairs) > 0:
    test_img_path = test_pairs[0][0]
    img = cv2.imread(test_img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    print(f"\nRunning inference on: {Path(test_img_path).name}")
    results = model.predict(img_rgb, verbose=False)

    print(f"OK Inference complete")
    print(f"  Predictions: {len(results)}")

    # Test conversion to metrics format
    conversion = yolo_result_to_metrics_input(results[0], test_img_path, "smoke_test")
    assert conversion["success"], f"Conversion failed: {conversion['error']}"
    print(f"OK YOLO result conversion works")

## Smoke Test Summary

In [ ]:
# Generate smoke test report
print("\n" + "=" * 80)
print("SMOKE TEST SUMMARY")
print("=" * 80)

smoke_test_report = {
    "test_type": "ablation_4_unified_smoke_test",
    "runtime_environment": "CPU",
    "dataset": "TINY (2 test, 4 train)",
    "tests_passed": {
        "1_ddi_split_logic": True,
        "2_4a_dataset_creation": True,
        "3_schema_validation": True,
        "4_completion_markers": True,
        "5_yolo_initialization": True,
    },
    "validation_checks": {
        "ddi_split_correct": len(ddi_train_paths) == 514 and len(ddi_test_paths) == 129,
        "metrics_schema_valid": expected_keys.issubset(actual_keys),
        "completion_marker_works": completion_4a.exists(),
        "yolo_inference_works": True,
    },
    "ready_for_production": all(
        [len(ddi_train_paths) == 514, expected_keys.issubset(actual_keys), completion_4a.exists()]
    ),
}

with open(OUTPUT_PATH / "smoke_test_report.json", "w") as f:
    json.dump(smoke_test_report, f, indent=2)

print("\nOK ALL SMOKE TESTS PASSED")
print(f"\nReport saved: {OUTPUT_PATH / 'smoke_test_report.json'}")

if smoke_test_report["ready_for_production"]:
    print("\n" + "=" * 80)
    print("OK READY FOR PRODUCTION DEPLOYMENT")
    print("=" * 80)
else:
    print("\nWARNING NOT READY - Fix failed tests before deployment")

print(f"\nValidation checks:")
for check, passed in smoke_test_report["validation_checks"].items():
    status = "OK" if passed else "FAIL"
    print(f"  {status} {check}")